# NanoGPT walkthrough

Contains walkthrough code from my own go-through of Karpathy's [nanoGPT video](https://www.youtube.com/watch?v=kCc8FmEb1nY). 

Reference Jupyter notebook: https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing

## Loading the dataset

Here, we use the Tiny Shakespeare dataset.

In [71]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# (urllib instead of wget — wget is often missing on macOS / minimal environments)
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

('input.txt', <http.client.HTTPMessage at 0x112c33e60>)

In [72]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [73]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


## Model 1: Creating a very simple Bigram model

To start, we'll first create a *very simple* Bigram model.

### What is a bigram model?

A bigram model is a very simple LM. A *bigram* is an ordered pair of consecutive symbols.

For example, in the string "bigram", you get the following bigrams: ["bi", "ig", "gr", "ra", "am"].

A bigram model works the following way: the probability of the next symbol depends on the symbol before it.

This is a very simple model, with many obvious caveats, but it's good for starting to work on language models. We can set up the same interface that we'd need for larger models, but we create the smallest setup possible.

### Setting up our tokenizer

First, we need to get all the characters in the text. This lets us build our vocabulary.

In [74]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [75]:
print(''.join(chars))
print(f"Total size of our vocabulary: {vocab_size}")


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Total size of our vocabulary: 65


#### Creating the mapping from characters to integers

Now we need to map the characters to integers.

In [76]:
string_to_int = { ch: i for i, ch in enumerate(chars) }
int_to_string = { i: ch for i, ch in enumerate(chars) }

In [77]:
def encode(string_input: str) -> list[int]:
    """Encodes a string into a list of integers using
    the mapping from characters to integers.
    """
    return [string_to_int[c] for c in string_input]

def decode(int_input: list[int]) -> str:
    """Decodes a list of integers into a string using
    the mapping from integers to characters."""
    return ''.join([int_to_string[i] for i in int_input])

In [78]:
test_string = "hello"
encoded_string = encode(test_string)
decoded_string = decode(encoded_string)
print(f"Encoded string: {encoded_string}")
print(f"Decoded string: {decoded_string}")


Encoded string: [46, 43, 50, 50, 53]
Decoded string: hello


#### Creating the tokenizer

We now wrap up that logic into a `Tokenizer` class.

In [79]:
class Tokenizer:
    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.stoi = { ch: i for i, ch in enumerate(chars) }
        self.itos = { i: ch for i, ch in enumerate(chars) }

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, l):
        return ''.join([self.itos[i] for i in l])

tokenizer = Tokenizer(vocab_size)

### Setting up train/test data

Now that we have a tokenizer, let's use it to set up our train/test data.

#### Applying the tokenizer on the text

In [80]:
encoded_text: list[int] = tokenizer.encode(text)

We see the first 25 characters here and their encoded indices.

In [81]:
num_sample_chars = 25
print(f"First {num_sample_chars} characters: {text[:num_sample_chars]}")
print(encoded_text[:num_sample_chars])

First 25 characters: First Citizen:
Before we 
[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1]


#### Turning the encoded text into PyTorch arrays

We now use PyTorch to represent the integer arrays. For our models, we'll use PyTorch as it's the way for us to do tensor arithmetic for the models.

In [82]:
import torch

In [83]:
encoded_text_tensor: torch.Tensor = torch.tensor(
    encoded_text, dtype=torch.long
)
print(f"Shape of encoded text tensor: {encoded_text_tensor.shape}")
print(f"First {num_sample_chars} characters: {text[:num_sample_chars]}")
print(encoded_text_tensor[:num_sample_chars])


Shape of encoded text tensor: torch.Size([1115394])
First 25 characters: First Citizen:
Before we 
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1])


Our text is ~1M characters long (1,115,394) and now we've represented it as a one-dimensional PyTorch tensor with 1,115,394 elements.

#### Creating train/test sets

We can now create train/test splits for the dataset. Let's use a 80/20 split.

We also want to introduce two concepts: (1) context length and (2) batch size.

##### Context length (block_size)

The context length (`context_length`) is how many consecutive tokens the model sees in one forward pass when you train/generate. The training data is a long stream of tokens, and we want to create "chunks" of context and use that. For example, if we have the text "this is a great day" and a context length of 8, we'd take the "this is " as one chunk (and then the next chunk would be one character shifted over, " his is a").

If you increase the context length, the model can use more history to predict the next token, but memory and compute grow linearly in sequence length (and attention, which we'll review later on, is quadratic in length).

Typical values can be some multiple of 2, e.g., 256, 1024, etc., but for pedagogical purposes we'll start with a value of 8.

We'll also call this the `block_size`, since we'll see later on that this length will determine the length of an individual sample in a batch. To give this a name consistent with the `batch_size` parameter, we'll call this `block_size`.

##### Batch size

Batch size (`batch_size`) tells us how many independent sequences (each of length up to `context_length`) are processed together in one forward/backward pass on the device.

An advanatage of GPUs is that the work can be very parallelizable; we can do large batches in one go, hence why we have the concept of a batch size. A larger batch means more parallel work per step and often more stable gradients (since for the gradient update step, we get more samples), but requires more GPU memory (and GPUs already don't have much memory, by design, because they're designed for computation, not memory).

##### How these fit together

Typically, the input to a model is a micro-batch of shape `(batch size, block_size)`, which we'll abbreviate as `(B,T)`:

- B = batch size = "how many chunks at once"
- T = context length = block size = "how many tokens per chunk"



### Putting these together in a Dataloader

We now set up a `Dataloader` class. This combines the previous concepts that we have before:

- We take as input the text.
- We tokenize it.
- We transform to PyTorch tensors.
- We create the train/test split of texts.
- We create a single block of text with a given `block_size` (aka context length)
- We create the batches, with shape `(batch_size, block_size)`.

In [84]:
class Dataloader():

    def __init__(
        self,
        batch_size: int,
        block_size: int,
        train_proportion: float,
        text: str,
        tokenizer: Tokenizer
    ):
        self.batch_size = batch_size
        self.block_size = block_size
        self.train_proportion = train_proportion
        self.text = text
        self.tokenizer = tokenizer


        self.train_text: str = ""
        self.val_text: str = ""
        self.train_batches: list[dict] = []
        self.val_batches: list[dict] = []

        # list of text -> int mappings. Each char is mapped to an int.
        self.train_string_to_int: list[int] = []
        self.val_string_to_int: list[int] = []

        # we take the list of ints and transform them into a tensor. This
        # tensor is what's passed into the model (albeit in batches).
        self.tensor_train_string_to_int: torch.Tensor = torch.tensor([])
        self.tensor_val_string_to_int: torch.Tensor = torch.tensor([])

        # get latest requested batch, if any.
        self.latest_train_batch_idx = -1
        self.latest_val_batch_idx = -1

        # we then transform this to a tensor list of ints.

    def tokenize(self) -> None:
        """Transforms the text into a tensor list of integers."""
        self._tokenize_to_int()
        self._transform_to_tensor()

    def _tokenize_to_int(self) -> None:
        """Tokenizes the text into a list of integers."""
        self.train_string_to_int = self.tokenizer.encode(self.train_text)
        self.val_string_to_int = self.tokenizer.encode(self.val_text)

    def _transform_to_tensor(self) -> None:
        """Transforms the list of integers into a tensor."""
        self.tensor_train_string_to_int = torch.tensor(self.train_string_to_int, dtype=torch.long)
        self.tensor_val_string_to_int = torch.tensor(self.val_string_to_int, dtype=torch.long)

    def define_train_test_splits(self) -> None:
        """Defines the train/test splits of the text."""

        train_size = int(self.train_proportion * len(self.text))
        val_size = len(self.text) - train_size
        self.train_text = self.text[:train_size]
        self.val_text = self.text[train_size:train_size + val_size]
        
        print(f"Train text length: {len(self.train_text)}")
        print(f"Val text length: {len(self.val_text)}")

    def create_batches(self) -> None:
        """Creates the train/test batches."""
        self._create_train_batches()
        self._create_val_batches()

    def _create_train_batches(self) -> None:
        """Each batch: x and y shaped (batch_size, block_size)."""
        self.train_batches.clear()
        t = self.tensor_train_string_to_int
        span = self.batch_size * self.block_size
        if len(t) < span + 1:
            raise ValueError("train split too short for this batch_size and block_size")

        for i in range(0, len(t) - span, span):
            xs: list[torch.Tensor] = []
            ys: list[torch.Tensor] = []
            texts: list[str] = []
            for b in range(self.batch_size):
                start = i + b * self.block_size
                xs.append(t[start : start + self.block_size])
                ys.append(t[start + 1 : start + self.block_size + 1])
                texts.append(self.train_text[start : start + self.block_size])

            self.train_batches.append(
                {
                    "original_text": texts,
                    "x": torch.stack(xs),
                    "y": torch.stack(ys),
                }
            )

        print(f"Created {len(self.train_batches)} train batches (each x,y is {self.batch_size} x {self.block_size})")

    def _create_val_batches(self) -> None:
        """Same layout as train, on the validation split."""
        self.val_batches.clear()
        t = self.tensor_val_string_to_int
        span = self.batch_size * self.block_size
        if len(t) < span + 1:
            raise ValueError("val split too short for this batch_size and block_size")

        for i in range(0, len(t) - span, span):
            xs: list[torch.Tensor] = []
            ys: list[torch.Tensor] = []
            texts: list[str] = []
            for b in range(self.batch_size):
                start = i + b * self.block_size
                xs.append(t[start : start + self.block_size])
                ys.append(t[start + 1 : start + self.block_size + 1])
                texts.append(self.val_text[start : start + self.block_size])

            self.val_batches.append(
                {
                    "original_text": texts,
                    "x": torch.stack(xs),
                    "y": torch.stack(ys),
                }
            )

        print(f"Created {len(self.val_batches)} validation batches (each x,y is {self.batch_size} x {self.block_size})")

    def get_train_batch(self, idx: int | None = None) -> dict:
        if idx is not None:
            return self.train_batches[idx]
        self.latest_train_batch_idx = (self.latest_train_batch_idx + 1) % len(self.train_batches)
        return self.train_batches[self.latest_train_batch_idx]

    def get_val_batch(self, idx: int | None = None) -> dict:
        if idx is not None:
            return self.val_batches[idx]
        self.latest_val_batch_idx = (self.latest_val_batch_idx + 1) % len(self.val_batches)
        return self.val_batches[self.latest_val_batch_idx]


Now we initialize the dataloader

In [85]:
torch.manual_seed(42)

batch_size = 4
block_size = 8
train_proportion = 0.8 # 80/20 split

vocab_size = len(chars)
tokenizer = Tokenizer(vocab_size)

In [86]:
loader = Dataloader(
    batch_size=batch_size,
    block_size=block_size,
    train_proportion=train_proportion,
    text=text,
    tokenizer=tokenizer
)
loader.define_train_test_splits()
loader.tokenize()
loader.create_batches()

Train text length: 892315
Val text length: 223079
Created 27884 train batches (each x,y is 4 x 8)
Created 6971 validation batches (each x,y is 4 x 8)


When we run the model, we pass in a batch. Let's see what one batch looks like.

In [87]:
result = loader.get_train_batch()
original_text = result["original_text"]
x = result["x"]
y = result["y"]

print(f"Original text: {original_text}")
print(f"x: {x}")
print(f"x.shape: {x.shape}")
print(f"y: {y}")
print(f"y.shape: {y.shape}")

Original text: ['First Ci', 'tizen:\nB', 'efore we', ' proceed']
x: tensor([[18, 47, 56, 57, 58,  1, 15, 47],
        [58, 47, 64, 43, 52, 10,  0, 14],
        [43, 44, 53, 56, 43,  1, 61, 43],
        [ 1, 54, 56, 53, 41, 43, 43, 42]])
x.shape: torch.Size([4, 8])
y: tensor([[47, 56, 57, 58,  1, 15, 47, 58],
        [47, 64, 43, 52, 10,  0, 14, 43],
        [44, 53, 56, 43,  1, 61, 43,  1],
        [54, 56, 53, 41, 43, 43, 42,  1]])
y.shape: torch.Size([4, 8])


In [88]:
print("Original text (one string per batch row):")
for row, s in enumerate(original_text):
    print(f"  row {row}: {s!r}")
print(f"x.shape: {x.shape}")
print(f"y.shape: {y.shape}")

Original text (one string per batch row):
  row 0: 'First Ci'
  row 1: 'tizen:\nB'
  row 2: 'efore we'
  row 3: ' proceed'
x.shape: torch.Size([4, 8])
y.shape: torch.Size([4, 8])


Things to notice:

- Our batch size of 4 represents the 4 strings that we're trying to learn and label in parallel. For each string, we take 8 tokens. 
- The labels for `y` are the same as the tokens for `x`, just shifted 1 over. This is the *next-token prediction task* in language models.
- Notice that each string is independent of other strings; we don't have any overlaps between any strings. In practice, we would have some overlapping, but it's OK here.

### Creating and training the Bigram language model

Now that we have a `Dataloader` to load the data, let's create the Bigram language model

#### Defining the Bigram language model

#### Training the Bigram language model

## Model 2: Transformer

### Transformer setup

We can decompose GPT into a series of building blocks. At heart, GPT goes from token IDs -> token embedding + position embedding -> dropout -> `n` transformer blocks -> a final LayerNorm -> and then a linear head over the vocabulary (logits).

In [89]:
import torch.nn as nn

In [ ]:
class LayerNorm(nn.Module):
    """Normalizes activations over the last dimension, for
    training stability. Using this instead of nn.LayerNorm
    since GPT-2 has no bias in the LayerNorm, but the PyTorch
    built-in version has it."""
    pass

In [ ]:
class CausalSelfAttention(nn.Module):
    
    pass

In [ ]:
class MLP(nn.Module):
    pass

In [ ]:
class TransformerBlock(nn.Module):
    pass

In [ ]:
class GPT(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pass

    def generate(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        pass

#### Defining the GPT config class

We also, like in the original NanoGPT implemmentation, set up a [GPTConfig](https://github.com/karpathy/nanoGPT/blob/master/model.py#L109) class.

### Training module

notes from nanoGPT video to consider:
- 